In [1]:
# 1. import rasters
import numpy as np, rasterio

BASIN_DDEM = "../data/rasters/canwell_nbasinlong_ddem_2010_2000.tif"
BASIN_ELEV = "../data/rasters/canwell_nbasinlong_elev2000.tif"
BASIN_NDVI = "../data/rasters/canwell_nbasinlong_ndvi2000.tif"
BASIN_LST  = "../data/rasters/canwell_nbasinlong_lst2000.tif"

def load(path):
    with rasterio.open(path) as s:
        a = s.read(1).astype(np.float32); nod = s.nodata; tr = s.transform
    a[(~np.isfinite(a)) | (a == nod)] = np.nan # set all inf or nodata to NaN
    return a, tr

ddem, transform = load(BASIN_DDEM)
elev, _         = load(BASIN_ELEV) # don't need transform again
ndvi, _         = load(BASIN_NDVI)
lst,  _         = load(BASIN_LST)
H, W = ddem.shape
valid = ~np.isnan(ddem)            # basin cells outside cutline are NaN

a, b, c, d, e, f = transform.a, transform.b, transform.c, transform.d, transform.e, transform.f
print("grid", (H,W), "| basin cells:", int(valid.sum()), "| resolution", (a, -e))

grid (1545, 2710) | basin cells: 835641 | resolution (5.0, 5.0)


In [2]:
# 2. graph pyramid --- successive nan-aware block poolings

import warnings; warnings.filterwarnings("ignore", category=RuntimeWarning)

def block_pool(arr, f):
    H, W = arr.shape
    Hp, Wp = -(-H // f) * f, -(-W // f) * f # round to integer multiple of f
    pad = np.full((Hp, Wp), np.nan, np.float32); pad[:H, :W] = arr
    pad = pad.reshape(Hp // f, f, Wp // f, f)    # take every f cells, stack "vertically"
    mean = np.nanmean(pad, axis=(1,3)).astype(np.float32)  # average over the vertically stacked cells
    frac = np.isfinite(pad).mean(axis=(1,3)).astype(np.float32) # fractional basin coverage of each new block
    return mean, frac

In [3]:
# 2.1 build levels
F = [3, 4, 15]      # 5->15, 15->60, 60->900 m
levels = {}
src_d, src_e = ddem, elev
src_n, src_l = ndvi, lst
res = 5.0
for L, fac in enumerate(F):
    dd, fr = block_pool(src_d, fac)
    el, _  = block_pool(src_e, fac)
    n, _  = block_pool(src_n, fac)
    l, _  = block_pool(src_l, fac)
    res *= fac                     # resolution * (new) factor
    levels[L] = dict(ddem=dd, elev=el, ndvi=n, lst=l, frac=fr, res=res, fac=fac)
    src_d, src_e = dd, el
    src_n, src_l = n, l

for L in levels:
    v = np.isfinite(levels[L]["ddem"])
    print(f"L{L} res {int(levels[L]['res'])}m grid {levels[L]['ddem'].shape} nodes {int(v.sum()):,}")

L0 res 15m grid (515, 904) nodes 93,759
L1 res 60m grid (129, 226) nodes 6,115
L2 res 900m grid (9, 16) nodes 49


In [4]:
# 3. assemble node table across levels
FEATS_POOL = ["ddem", "elev", "ndvi", "lst"]
c0, f0 = c, f                               # left-edge x, top-edge y from cell 1

node_level=[]; node_row=[]; node_col=[]; node_x=[]; node_y=[]; node_frac=[]
feats={k:[] for k in FEATS_POOL}; idx_grid={}; gid=0

for L in sorted(levels):
    lev=levels[L]; resL=lev["res"]
    rr, cc = np.where(np.isfinite(lev["ddem"]))
    nn=rr.size
    ig=np.full(lev["ddem"].shape, -1, np.int64); ig[rr, cc]=np.arange(gid, gid+nn)
    idx_grid[L]=ig # assign (global) index to each node of a given level
    node_level.append(np.full(nn, L, np.int8))
    node_row.append(rr.astype(np.int32)); node_col.append(cc.astype(np.int32))
    node_x.append(c0 + resL*(cc+0.5)); node_y.append(f0 - resL*(rr+0.5))
    node_frac.append(lev["frac"][rr, cc])
    for k in FEATS_POOL: feats[k].append(lev[k][rr, cc])
    gid += nn

# given a (global) node index, can look up its level, row, col, true x,y, fractional coverage
node_level=np.concatenate(node_level); node_row=np.concatenate(node_row); node_col=np.concatenate(node_col)
node_x=np.concatenate(node_x); node_y=np.concatenate(node_y); node_frac=np.concatenate(node_frac)

for k in FEATS_POOL: feats[k]=np.concatenate(feats[k]).astype(np.float32) # raster values by node index

N = gid # total number of nodes -- i.e. sequential node index

print("total basin nodes:", N, "| per level:", {L:int((node_level==L).sum()) for L in levels})

total basin nodes: 99923 | per level: {0: 93759, 1: 6115, 2: 49}


In [5]:
# 4. intra-level 8-connectivity
offsets = [(0,1),(1,0),(1,1),(1,-1)]    # each undirected edge once
src_1, dst_1 = [], []
for L in sorted(levels):
    ig = idx_grid[L]; H1, W1 = ig.shape
    for dr, dc in offsets:
        aa = ig[max(0,-dr):H1-max(0,dr), max(0, -dc):W1-max(0,dc)]
        bb = ig[max(0, dr):H1-max(0,-dr), max(0,dc):W1-max(0,-dc)]
        m = (aa>0) & (bb>0) # valid indices only
        src_1.append(aa[m]); dst_1.append(bb[m])

src = np.concatenate(src_1); dst = np.concatenate(dst_1)
intra_edge = np.vstack([np.concatenate([src, dst]),
                        np.concatenate([dst, src])]).astype(np.int64)
print("intra-level  undirected:", src.size, "| directed:", intra_edge.shape[1])


intra-level  undirected: 395613 | directed: 791226


In [6]:
# 5. inter-level connectivity by containment (parent) edges
# each finer node links to single coarser node that contains it
# by integer-dividing grid position by pooling factor
# symmetrized (bi-directional, undirected) so information flows both ways
# note: "parent" indicates hierarchy in the graph, NOT "source" of pooling
# a hierarchical "parent" cell was generated by pooling finer cells
#   of the level below
# all "child" cells with same "parent" link to that parent

isrc_1, idst_1 = [], []; dropped = 0
for L in sorted(levels):
    if L+1 not in levels:
        continue
    fac = F[L+1]  # factor that pooled level L+1 from level L
    ig, pg = idx_grid[L], idx_grid[L+1] # indices and pooled indices
    rr, cc = np.where(ig >= 0)
    child = ig[rr,cc] # "originating" (finer) cells
    parent = pg[rr//fac, cc//fac] # each cell's "parent" (pooled) cell
    ok = parent >= 0; dropped += int((~ok).sum())
    isrc_1.append(child[ok]); idst_1.append(parent[ok])

isrc = np.concatenate(isrc_1); idst = np.concatenate(idst_1)
inter_edge = np.vstack([np.concatenate([isrc, idst]),
                        np.concatenate([idst, isrc])]).astype(np.int64)
print("inter-level undirected:", isrc.size, "| directed:", inter_edge.shape[1], "| dropped:", dropped)

inter-level undirected: 99874 | directed: 199748 | dropped: 0


In [8]:
# 6. save graph and metadata
import json, datetime

feat_names = FEATS_POOL + ["frac"]
Xbasin = np.column_stack([feats[k] for k in FEATS_POOL] + [node_frac]).astype(np.float32)

edge_index = np.concatenate([intra_edge, inter_edge], axis=1)
edge_type  = np.concatenate([np.zeros(intra_edge.shape[1], np.int8),
                             np.ones(inter_edge.shape[1], np.int8)]) # 0=intra, 1=inter

np.savez_compressed("../data/constructs/basin_mushgraph.npz",
                    level=node_level, row=node_row, col=node_col,
                    x=node_x.astype(np.float64), y=node_y.astype(np.float64),
                    X=Xbasin, feat_names=np.array(feat_names),
                    edge_index=edge_index, edge_type=edge_type,
                    intra_edge=intra_edge, inter_edge=inter_edge,
                    shape=np.array([H,W], np.int64),
                    transform=np.array([a, b, c, d, e, f], np.float64),
                    pool_factors=np.array(F, np.int64),
                    level_res=np.array([levels[L]["res"] for L in sorted(levels)], np.float64))

meta = {
    "created":datetime.date.today().isoformat(),
    "source_ddem": BASIN_DDEM, "source_elev": BASIN_ELEV,
    "grid_5m": [int(H), int(W)], "epsg": 32606,
    "levels": {int(L): {"res_m": float(levels[L]["res"]),
                        "grid": list(map(int, levels[L]["ddem"].shape)),
                        "nodes": int((node_level==L).sum())} for L in sorted(levels)},
    "pool_factors": F, "n_nodes":int(N),
    "edges": {"intra_undirected": intra_edge.shape[1]//2,
              "inter_undirected": inter_edge.shape[1]//2},
    "node_features": feat_names,
    "notes": "multiscale basin pyramid; nan-aware block-mean pooling; intra-connectivity=8-neighbor"
             "inter-connectivity=containment parent edges; x,y = cell-center map coords in EPSG:32606; "
             "frac=basin coverage of pooled cell; mushgraph: multi-scale hierarchical graph",
}

with open("../data/constructs/basin_mushgraph.meta.json", "w") as fh:
    json.dump(meta, fh, indent=2)
print("saved basin_mushgraph.npz | nodes:", N, "| directed edges", edge_index.shape[1], "| features", feat_names)

saved basin_mushgraph.npz | nodes: 99923 | directed edges 990974 | features ['ddem', 'elev', 'ndvi', 'lst', 'frac']
